# Urban Expansion and Land Surface Temperature in Elmina

**Research question:**  
**How is urban expansion associated with changes in land surface temperature in the Elmina urban expansion corridor, Greater Kuala Lumpur, between 2004 and 2025?**

This notebook compares three ways of identifying urbanisation-related change:

1. an interpretable NDVI/NDBI threshold baseline,
2. a pretrained zero-shot ChangeStar transfer experiment,
3. a locally trained Siamese CNN using bi-temporal Landsat patches.

The resulting change patterns are compared with scene-relative Land Surface Temperature (LST) change.

> **Environment note:** package installation cells are intentionally excluded from the analytical workflow. Required Python packages include `numpy`, `pandas`, `matplotlib`, `rasterio`, `pystac-client`, `planetary-computer`, `torch`, `scikit-learn`, and `torchange`.

## 1. Project setup and Landsat scene selection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from pathlib import Path
from pystac_client import Client
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from matplotlib.colors import TwoSlopeNorm, ListedColormap
from matplotlib.patches import Patch

import planetary_computer


In [ ]:
# --------------------------------------------------
# PROJECT CONFIGURATION
# --------------------------------------------------

STUDY_AREA = "Elmina, Greater Kuala Lumpur"

BBOX = [
    101.50,   # west
    3.16,     # south
    101.54,   # east
    3.20      # north
]

DATE_T1 = "2004-05-31"
DATE_T2 = "2025-05-17"

PLATFORM_T1 = "landsat-5"
PLATFORM_T2 = "landsat-9"

print("Study area:", STUDY_AREA)
print("T1:", DATE_T1, PLATFORM_T1)
print("T2:", DATE_T2, PLATFORM_T2)
print("Bounding box:", BBOX)

In [ ]:
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

In [ ]:
def get_landsat_scene(date, platform, bbox):

    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=f"{date}/{date}",
        query={
            "platform": {"eq": platform}
        }
    )

    items = list(search.items())

    if len(items) == 0:
        raise ValueError(
            f"No Landsat scene found for {date} and {platform}"
        )

    return items[0]

In [ ]:
scene_t1 = get_landsat_scene(
    DATE_T1,
    PLATFORM_T1,
    BBOX
)

scene_t2 = get_landsat_scene(
    DATE_T2,
    PLATFORM_T2,
    BBOX
)

print("T1:", scene_t1.id)
print("T2:", scene_t2.id)

## 2. Quality control and common spatial grid

Cloud/cloud-shadow contamination is checked within the Elmina ROI using the Landsat `QA_PIXEL` band.  
The 2004 and 2025 scenes must also share the same CRS, 30 m resolution, shape and pixel transform before pixelwise change analysis.

In [ ]:
def cloud_percentage_roi(item, bbox):
    """
    Calculate the percentage of cloud-affected pixels
    within the study area using the Landsat QA_PIXEL band.
    """

    # Sign Planetary Computer asset URLs
    signed_item = planetary_computer.sign(item)

    qa_url = signed_item.assets["qa_pixel"].href

    with rasterio.open(qa_url) as src:

        # Transform study-area coordinates to raster CRS
        left, bottom, right, top = transform_bounds(
            "EPSG:4326",
            src.crs,
            bbox[0],
            bbox[1],
            bbox[2],
            bbox[3]
        )

        # Select only the study area
        window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform
        )

        qa = src.read(
            1,
            window=window,
            masked=True
        )

    values = qa.data
    valid = ~np.ma.getmaskarray(qa)

    # Landsat QA_PIXEL flags
    dilated_cloud = (values & (1 << 1)) != 0
    cloud = (values & (1 << 3)) != 0
    cloud_shadow = (values & (1 << 4)) != 0

    cloudy = dilated_cloud | cloud | cloud_shadow

    # Landsat 8/9 additionally use the cirrus flag
    platform = item.properties.get("platform", "")

    if platform in ["landsat-8", "landsat-9"]:
        cirrus = (values & (1 << 2)) != 0
        cloudy = cloudy | cirrus

    cloudy_valid = cloudy & valid

    return cloudy_valid.sum() / valid.sum() * 100

In [ ]:
cloud_t1 = cloud_percentage_roi(scene_t1, BBOX)
cloud_t2 = cloud_percentage_roi(scene_t2, BBOX)

print(f"T1 cloud-affected pixels: {cloud_t1:.2f}%")
print(f"T2 cloud-affected pixels: {cloud_t2:.2f}%")

In [ ]:
def read_reflectance_band(item, bbox, asset_name):
    """
    Read one Landsat surface-reflectance band
    within the study area and apply Collection 2
    Level 2 reflectance scaling.
    """

    signed_item = planetary_computer.sign(item)
    url = signed_item.assets[asset_name].href

    with rasterio.open(url) as src:

        left, bottom, right, top = transform_bounds(
            "EPSG:4326",
            src.crs,
            bbox[0],
            bbox[1],
            bbox[2],
            bbox[3]
        )

        window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform
        )

        band = src.read(
            1,
            window=window,
            masked=True
        ).astype("float32")

    # Landsat Collection 2 Level 2 scaling
    reflectance = band * 0.0000275 - 0.2

    # Convert masked pixels to NaN
    return reflectance.filled(np.nan)

In [ ]:
def get_grid_info(item, bbox, asset_name="red"):
    """
    Inspect the spatial grid of a Landsat band
    within the study area.
    """

    signed_item = planetary_computer.sign(item)
    url = signed_item.assets[asset_name].href

    with rasterio.open(url) as src:

        left, bottom, right, top = transform_bounds(
            "EPSG:4326",
            src.crs,
            bbox[0],
            bbox[1],
            bbox[2],
            bbox[3]
        )

        window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        return {
            "crs": str(src.crs),
            "resolution": src.res,
            "shape": (
                int(window.height),
                int(window.width)
            ),
            "transform": src.window_transform(window),
            "bounds": rasterio.windows.bounds(
                window,
                src.transform
            )
        }

In [ ]:
grid_t1 = get_grid_info(scene_t1, BBOX)
grid_t2 = get_grid_info(scene_t2, BBOX)

print("T1:", grid_t1)
print("T2:", grid_t2)

print()
print("Same CRS:", grid_t1["crs"] == grid_t2["crs"])
print(
    "Same resolution:",
    np.allclose(
        grid_t1["resolution"],
        grid_t2["resolution"]
    )
)
print("Same shape:", grid_t1["shape"] == grid_t2["shape"])
print(
    "Same transform:",
    np.allclose(
        tuple(grid_t1["transform"]),
        tuple(grid_t2["transform"])
    )
)

## 3. Index-based change-detection baseline

NDVI is used to represent vegetation change, while NDBI provides a complementary built-up / bare-soil signal.  
The baseline identifies candidate urbanisation where NDVI decreases and NDBI increases.

In [ ]:
def calculate_ndvi(item, bbox):
    """
    Calculate NDVI:
    (NIR - Red) / (NIR + Red)
    """

    red = read_reflectance_band(item, bbox, "red")
    nir = read_reflectance_band(item, bbox, "nir08")

    denominator = nir + red

    ndvi = np.where(
        denominator != 0,
        (nir - red) / denominator,
        np.nan
    )

    return ndvi


def calculate_ndbi(item, bbox):
    """
    Calculate NDBI:
    (SWIR - NIR) / (SWIR + NIR)
    """

    nir = read_reflectance_band(item, bbox, "nir08")
    swir = read_reflectance_band(item, bbox, "swir16")

    denominator = swir + nir

    ndbi = np.where(
        denominator != 0,
        (swir - nir) / denominator,
        np.nan
    )

    return ndbi

In [ ]:
ndvi_t1 = calculate_ndvi(scene_t1, BBOX)
ndvi_t2 = calculate_ndvi(scene_t2, BBOX)

ndbi_t1 = calculate_ndbi(scene_t1, BBOX)
ndbi_t2 = calculate_ndbi(scene_t2, BBOX)

print("Mean NDVI T1:", np.nanmean(ndvi_t1))
print("Mean NDVI T2:", np.nanmean(ndvi_t2))

print("Mean NDBI T1:", np.nanmean(ndbi_t1))
print("Mean NDBI T2:", np.nanmean(ndbi_t2))

In [ ]:
ndvi_change = ndvi_t2 - ndvi_t1
ndbi_change = ndbi_t2 - ndbi_t1

print("Mean NDVI change:", np.nanmean(ndvi_change))
print("Mean NDBI change:", np.nanmean(ndbi_change))

In [ ]:
def detect_candidate_urbanisation(
    ndvi_change,
    ndbi_change,
    ndvi_threshold=-0.20,
    ndbi_threshold=0.10
):
    """
    Detect candidate urbanisation based on
    decreasing NDVI and increasing NDBI.
    """

    valid = (
        np.isfinite(ndvi_change) &
        np.isfinite(ndbi_change)
    )

    candidate = (
        (ndvi_change <= ndvi_threshold) &
        (ndbi_change >= ndbi_threshold) &
        valid
    )

    return candidate, valid

In [ ]:
urbanisation_candidate, valid_pixels = detect_candidate_urbanisation(
    ndvi_change,
    ndbi_change
)

urbanisation_percentage = (
    urbanisation_candidate.sum()
    / valid_pixels.sum()
    * 100
)

print(
    f"Candidate urbanisation area: "
    f"{urbanisation_percentage:.2f}%"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(ndvi_change)
axes[0].set_title("NDVI Change")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="ΔNDVI")

im1 = axes[1].imshow(ndbi_change)
axes[1].set_title("NDBI Change")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="ΔNDBI")

axes[2].imshow(urbanisation_candidate, cmap="gray")
axes[2].set_title("Initial Candidate Urbanisation")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 4. QGIS reference data and baseline validation

Reference polygons were digitised manually in QGIS by comparing the 2004 and 2025 imagery without using the model prediction as a guide.

- `1` = urbanisation change
- `0` = explicitly labelled no urbanisation change
- unlabelled pixels = NoData

The reference raster is aligned to the Landsat 30 m grid. Only explicitly labelled pixels are used for evaluation.

In [ ]:
def export_rgb_geotiff(item, bbox, filename):
    """
    Export a georeferenced true-colour Landsat RGB image
    for use in QGIS.
    """

    red = read_reflectance_band(item, bbox, "red")
    green = read_reflectance_band(item, bbox, "green")
    blue = read_reflectance_band(item, bbox, "blue")

    rgb = np.stack([red, green, blue])

    grid = get_grid_info(item, bbox, "red")

    with rasterio.open(
        filename,
        "w",
        driver="GTiff",
        height=rgb.shape[1],
        width=rgb.shape[2],
        count=3,
        dtype="float32",
        crs=grid["crs"],
        transform=grid["transform"]
    ) as dst:

        dst.write(rgb.astype("float32"))

        dst.set_band_description(1, "Red")
        dst.set_band_description(2, "Green")
        dst.set_band_description(3, "Blue")

    print("Saved:", filename)

The RGB export function above was used to prepare georeferenced 2004 and 2025 true-colour images for manual QGIS labelling.  
The export calls themselves are omitted from the final analytical run to avoid overwriting files.

In [ ]:
reference_path = Path("qgis_reference") / "elmina_reference_change_mask.tif"

print("Reference path:", reference_path)
print("Exists:", reference_path.exists())

with rasterio.open(reference_path) as src:
    reference_masked = src.read(1, masked=True)

    print("Shape:", reference_masked.shape)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Transform:", src.transform)
    print("NoData:", src.nodata)

labelled = ~reference_masked.mask
reference = reference_masked.data

values, counts = np.unique(
    reference_masked.compressed(),
    return_counts=True
)

print("\nLabel values:")
for value, count in zip(values, counts):
    print(value, count)

print("\nTotal labelled pixels:", labelled.sum())

In [ ]:
# Spatial distribution of labelled reference pixels

labelled = ~reference_masked.mask
reference = reference_masked.data

rows, cols = np.where(labelled)

labels = reference[labelled]

plt.figure(figsize=(8, 8))

plt.scatter(
    cols[labels == 0],
    rows[labels == 0],
    label="No urbanisation change",
    marker="s"
)

plt.scatter(
    cols[labels == 1],
    rows[labels == 1],
    label="Urbanisation change",
    marker="s"
)

plt.gca().invert_yaxis()

plt.xlabel("Raster column")
plt.ylabel("Raster row")
plt.title("Spatial Distribution of Reference Labels")

plt.legend()
plt.show()

In [ ]:
row_indices = np.indices(labelled.shape)[0]

SPLIT_ROW = 50
GAP = 5  # 5 Landsat pixels = 150 m

validation_mask = (
    labelled &
    (row_indices < SPLIT_ROW - GAP)
)

test_mask = (
    labelled &
    (row_indices > SPLIT_ROW + GAP)
)

print("Validation pixels:", validation_mask.sum())
print("Test-subset pixels:", test_mask.sum())
print("Unassigned labelled pixels:", labelled.sum() - validation_mask.sum() - test_mask.sum())


Threshold choice is based only on the northern validation subset. The southern subset is spatially separated and reported as a **test subset**.  
Because earlier exploratory development had already inspected aggregate reference performance, the test subset is not described as a strictly untouched held-out set.

In [ ]:
thresholds = [
    (-0.15, 0.05),
    (-0.20, 0.10),
    (-0.25, 0.15)
]

for ndvi_threshold, ndbi_threshold in thresholds:

    prediction_val = (
        (ndvi_change <= ndvi_threshold) &
        (ndbi_change >= ndbi_threshold)
    )

    tp = np.sum(
        validation_mask &
        (reference == 1) &
        prediction_val
    )

    tn = np.sum(
        validation_mask &
        (reference == 0) &
        (~prediction_val)
    )

    fp = np.sum(
        validation_mask &
        (reference == 0) &
        prediction_val
    )

    fn = np.sum(
        validation_mask &
        (reference == 1) &
        (~prediction_val)
    )

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else np.nan
    )

    accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )

    print(
        f"\nNDVI <= {ndvi_threshold}, "
        f"NDBI >= {ndbi_threshold}"
    )

    print("TP:", tp, "TN:", tn, "FP:", fp, "FN:", fn)
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 score:  {f1:.3f}")

In [ ]:
# --------------------------------------------------
# SPATIALLY SEPARATED TEST SUBSET
# --------------------------------------------------

FINAL_NDVI_THRESHOLD = -0.20
FINAL_NDBI_THRESHOLD = 0.10

final_prediction = (
    (ndvi_change <= FINAL_NDVI_THRESHOLD) &
    (ndbi_change >= FINAL_NDBI_THRESHOLD)
)

# Confusion matrix on the spatially separated test subset
tp_test = np.sum(
    test_mask &
    (reference == 1) &
    final_prediction
)

tn_test = np.sum(
    test_mask &
    (reference == 0) &
    (~final_prediction)
)

fp_test = np.sum(
    test_mask &
    (reference == 0) &
    final_prediction
)

fn_test = np.sum(
    test_mask &
    (reference == 1) &
    (~final_prediction)
)

# Metrics
accuracy_test = (
    (tp_test + tn_test) /
    (tp_test + tn_test + fp_test + fn_test)
)

precision_test = (
    tp_test / (tp_test + fp_test)
    if (tp_test + fp_test) > 0
    else np.nan
)

recall_test = (
    tp_test / (tp_test + fn_test)
    if (tp_test + fn_test) > 0
    else np.nan
)

f1_test = (
    2 * precision_test * recall_test /
    (precision_test + recall_test)
    if (precision_test + recall_test) > 0
    else np.nan
)

print("SPATIALLY SEPARATED TEST SUBSET")
print("-------------------")
print("TP:", tp_test)
print("TN:", tn_test)
print("FP:", fp_test)
print("FN:", fn_test)

print()
print(f"Accuracy:  {accuracy_test:.3f}")
print(f"Precision: {precision_test:.3f}")
print(f"Recall:    {recall_test:.3f}")
print(f"F1 score:  {f1_test:.3f}")


In [ ]:
# Final index-based urbanisation mask using the validation-selected thresholds
final_urbanisation_mask = (
    (ndvi_change <= FINAL_NDVI_THRESHOLD) &
    (ndbi_change >= FINAL_NDBI_THRESHOLD)
)

print(
    "Detected urbanisation pixels:",
    final_urbanisation_mask.sum()
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

index_confusion_matrix = np.array([
    [tn_test, fp_test],
    [fn_test, tp_test]
])

fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(index_confusion_matrix)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels([
    "Predicted no change",
    "Predicted urbanisation"
])

ax.set_yticklabels([
    "Reference no change",
    "Reference urbanisation"
])

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            index_confusion_matrix[i, j],
            ha="center",
            va="center",
            fontsize=14
        )

ax.set_title(
    "Spatial Test-Subset Confusion Matrix"
)

plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Prepare display raster
change_map = np.where(
    valid_pixels,
    final_urbanisation_mask.astype(float),
    np.nan
)

# 0 = no detected urbanisation change
# 1 = detected urbanisation change
cmap = ListedColormap([
    "lightgray",
    "firebrick"
])

fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(
    change_map,
    cmap=cmap,
    vmin=0,
    vmax=1
)

ax.set_title(
    "Final Urbanisation Change Detection\n"
    "Elmina, 2004–2025"
)

ax.axis("off")

legend_elements = [
    Patch(
        facecolor="lightgray",
        label="No detected urbanisation change"
    ),
    Patch(
        facecolor="firebrick",
        label="Detected urbanisation change"
    )
]

ax.legend(
    handles=legend_elements,
    loc="lower left"
)

plt.tight_layout()
plt.show()

### Baseline interpretation

The validation-selected NDVI/NDBI rule is retained as an interpretable remote-sensing baseline.  
Its spatially separated test-subset performance is reported alongside the GeoAI models below.

## 5. GeoAI transfer experiment: pretrained ChangeStar

A pretrained zero-shot, class-agnostic ChangeStar model is applied to the same bi-temporal RGB imagery.  
This is an exploratory transfer experiment rather than local model training. The model's generic change score is evaluated against the urbanisation-specific QGIS reference labels.

In [ ]:
import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from torchange.models.changen2 import s0_init_s1c1_changestar_vitb_1x256

changestar_model = s0_init_s1c1_changestar_vitb_1x256()
changestar_model.eval()

print(type(changestar_model))
print("Pretrained ChangeStar loaded successfully")

In [ ]:
# RGB reflectance for both dates
rgb_t1 = np.dstack([
    read_reflectance_band(scene_t1, BBOX, "red"),
    read_reflectance_band(scene_t1, BBOX, "green"),
    read_reflectance_band(scene_t1, BBOX, "blue")
]).astype(np.float32)

rgb_t2 = np.dstack([
    read_reflectance_band(scene_t2, BBOX, "red"),
    read_reflectance_band(scene_t2, BBOX, "green"),
    read_reflectance_band(scene_t2, BBOX, "blue")
]).astype(np.float32)

# Common 2–98 percentile stretch per channel, using both dates together
for c in range(3):
    both = np.concatenate([
        rgb_t1[:, :, c].ravel(),
        rgb_t2[:, :, c].ravel()
    ])

    lo = np.nanpercentile(both, 2)
    hi = np.nanpercentile(both, 98)

    rgb_t1[:, :, c] = np.clip(
        (rgb_t1[:, :, c] - lo) / (hi - lo + 1e-6), 0, 1
    )
    rgb_t2[:, :, c] = np.clip(
        (rgb_t2[:, :, c] - lo) / (hi - lo + 1e-6), 0, 1
    )

t1_tensor = torch.from_numpy(
    np.transpose(rgb_t1, (2, 0, 1))
).unsqueeze(0).float()

t2_tensor = torch.from_numpy(
    np.transpose(rgb_t2, (2, 0, 1))
).unsqueeze(0).float()

# ImageNet normalization
imagenet_mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(1, 3, 1, 1)

imagenet_std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(1, 3, 1, 1)

t1_tensor = (t1_tensor - imagenet_mean) / imagenet_std
t2_tensor = (t2_tensor - imagenet_mean) / imagenet_std

t1_resized = F.interpolate(
    t1_tensor, size=(256, 256),
    mode="bilinear", align_corners=False
)

t2_resized = F.interpolate(
    t2_tensor, size=(256, 256),
    mode="bilinear", align_corners=False
)

changestar_input = torch.cat(
    [t1_resized, t2_resized],
    dim=1
)

with torch.no_grad():
    changestar_output = changestar_model(changestar_input)

changestar_prob = F.interpolate(
    changestar_output.change_prediction,
    size=ndvi_change.shape,
    mode="bilinear",
    align_corners=False
)

changestar_prob_np = (
    changestar_prob.squeeze().cpu().numpy()
)

reference_valid_changestar = (
    labelled &
    np.isfinite(changestar_prob_np)
)

changestar_y_true = reference[
    reference_valid_changestar
]

changestar_y_score = changestar_prob_np[
    reference_valid_changestar
]

changestar_auc = roc_auc_score(
    changestar_y_true,
    changestar_y_score
)

changestar_urban_mean = np.mean(
    changestar_prob_np[
        reference_valid_changestar &
        (reference == 1)
    ]
)

changestar_nochange_mean = np.mean(
    changestar_prob_np[
        reference_valid_changestar &
        (reference == 0)
    ]
)

print(f"Mean score – urbanisation: {changestar_urban_mean:.3f}")
print(f"Mean score – no urbanisation change: {changestar_nochange_mean:.3f}")
print(f"ChangeStar ROC-AUC: {changestar_auc:.3f}")

### ChangeStar result

The pretrained zero-shot ChangeStar transfer produced **ROC-AUC = 0.497** against the urbanisation-specific QGIS labels, i.e. approximately random discrimination for this target.

This is interpreted as a transfer limitation rather than evidence that GeoAI is generally ineffective. Two important mismatches are present:

- substantial domain and spatial-resolution shift between high-resolution RGB pretraining data and 30 m Landsat imagery;
- ChangeStar predicts generic class-agnostic change, whereas the reference target is specifically **urbanisation change**.

ChangeStar is therefore retained as a documented transfer experiment but is not used as the primary urbanisation mask.

## 6. Locally supervised GeoAI: Siamese CNN

A compact Siamese CNN is trained directly on the Elmina reference data.  
For each labelled Landsat pixel, two 9×9 patches (2004 and 2025) are created using six reflectance bands per date: Blue, Green, Red, NIR, SWIR1 and SWIR2.

The same CNN encoder is applied to both dates; the classifier uses the absolute difference between the learned feature vectors.

In [ ]:
# Six spectral bands for the Siamese CNN
BANDS = [
    "blue",
    "green",
    "red",
    "nir08",
    "swir16",
    "swir22"
]

def build_multispectral_stack(item, bbox, bands):
    arrays = [
        read_reflectance_band(item, bbox, band)
        for band in bands
    ]

    # Shape: [bands, rows, columns]
    return np.stack(arrays, axis=0).astype(np.float32)


stack_t1 = build_multispectral_stack(
    scene_t1,
    BBOX,
    BANDS
)

stack_t2 = build_multispectral_stack(
    scene_t2,
    BBOX,
    BANDS
)

print("T1 shape:", stack_t1.shape)
print("T2 shape:", stack_t2.shape)

print("\nT1 band means:")
for band, value in zip(BANDS, np.nanmean(stack_t1, axis=(1, 2))):
    print(f"{band:8s}: {value:.4f}")

print("\nT2 band means:")
for band, value in zip(BANDS, np.nanmean(stack_t2, axis=(1, 2))):
    print(f"{band:8s}: {value:.4f}")

In [ ]:
PATCH_SIZE = 9
HALF = PATCH_SIZE // 2

patches_t1 = []
patches_t2 = []
patch_labels = []
patch_rows = []
patch_cols = []

rows, cols = np.where(labelled)

for row, col in zip(rows, cols):
    # 9x9 patch um den gelabelten Pixel
    patch_t1 = stack_t1[
        :,
        row-HALF:row+HALF+1,
        col-HALF:col+HALF+1
    ]

    patch_t2 = stack_t2[
        :,
        row-HALF:row+HALF+1,
        col-HALF:col+HALF+1
    ]

    # Nur vollständige und finite Patches behalten
    if (
        patch_t1.shape == (6, PATCH_SIZE, PATCH_SIZE)
        and patch_t2.shape == (6, PATCH_SIZE, PATCH_SIZE)
        and np.all(np.isfinite(patch_t1))
        and np.all(np.isfinite(patch_t2))
    ):
        patches_t1.append(patch_t1)
        patches_t2.append(patch_t2)
        patch_labels.append(int(reference[row, col]))
        patch_rows.append(row)
        patch_cols.append(col)

patches_t1 = np.stack(patches_t1).astype(np.float32)
patches_t2 = np.stack(patches_t2).astype(np.float32)
patch_labels = np.array(patch_labels, dtype=np.int64)
patch_rows = np.array(patch_rows)
patch_cols = np.array(patch_cols)

print("T1 patches:", patches_t1.shape)
print("T2 patches:", patches_t2.shape)

print("\nClass distribution:")
print("No urbanisation change:", np.sum(patch_labels == 0))
print("Urbanisation change:", np.sum(patch_labels == 1))
print("Total:", len(patch_labels))

In [ ]:
# räumliche Verteilung der CNN-Beispiele prüfen
for cls in [0, 1]:
    mask = patch_labels == cls

    print(f"\nClass {cls}")
    print("n:", mask.sum())
    print(
        "rows:",
        patch_rows[mask].min(),
        "to",
        patch_rows[mask].max()
    )
    print(
        "cols:",
        patch_cols[mask].min(),
        "to",
        patch_cols[mask].max()
    )

plt.figure(figsize=(7, 7))

plt.scatter(
    patch_cols[patch_labels == 0],
    patch_rows[patch_labels == 0],
    label="No urbanisation change",
    alpha=0.7
)

plt.scatter(
    patch_cols[patch_labels == 1],
    patch_rows[patch_labels == 1],
    label="Urbanisation change",
    alpha=0.7
)

plt.gca().invert_yaxis()
plt.xlabel("Column")
plt.ylabel("Row")
plt.title("Spatial distribution of CNN reference patches")
plt.legend()
plt.show()

In [ ]:
# Two spatial folds based on the four separated reference blocks

upper_fold = patch_rows < 50
lower_fold = patch_rows > 55

print("UPPER FOLD")
print("Total:", upper_fold.sum())
print("Class 0:", np.sum(patch_labels[upper_fold] == 0))
print("Class 1:", np.sum(patch_labels[upper_fold] == 1))

print("\nLOWER FOLD")
print("Total:", lower_fold.sum())
print("Class 0:", np.sum(patch_labels[lower_fold] == 0))
print("Class 1:", np.sum(patch_labels[lower_fold] == 1))

print("\nUnused pixels between folds:",
      np.sum(~upper_fold & ~lower_fold))

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# --------------------------------------------------
# Siamese CNN
# --------------------------------------------------
class SiameseChangeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Shared encoder for T1 and T2
        self.encoder = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Classification based on feature difference
        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )

    def forward(self, t1, t2):
        f1 = self.encoder(t1).flatten(1)
        f2 = self.encoder(t2).flatten(1)

        difference = torch.abs(f2 - f1)

        return self.classifier(difference).squeeze(1)


print(SiameseChangeCNN())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# --------------------------------------------------
# Fixed training settings
# --------------------------------------------------
EPOCHS = 60
LEARNING_RATE = 0.001

def normalize_from_training(train_t1, train_t2, test_t1, test_t2):
    """
    Calculate one mean/std per spectral band using TRAINING DATA ONLY.
    T1 and T2 are combined so temporal differences are preserved.
    """

    combined_train = np.concatenate(
        [train_t1, train_t2],
        axis=0
    )

    band_mean = combined_train.mean(axis=(0, 2, 3), keepdims=True)
    band_std = combined_train.std(axis=(0, 2, 3), keepdims=True)

    band_std[band_std < 1e-6] = 1.0

    train_t1_norm = (train_t1 - band_mean) / band_std
    train_t2_norm = (train_t2 - band_mean) / band_std

    test_t1_norm = (test_t1 - band_mean) / band_std
    test_t2_norm = (test_t2 - band_mean) / band_std

    return (
        train_t1_norm,
        train_t2_norm,
        test_t1_norm,
        test_t2_norm
    )


def train_and_test_fold(train_mask, test_mask, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)

    # ----------------------------------------------
    # Split
    # ----------------------------------------------
    train_t1 = patches_t1[train_mask]
    train_t2 = patches_t2[train_mask]
    y_train = patch_labels[train_mask]

    test_t1 = patches_t1[test_mask]
    test_t2 = patches_t2[test_mask]
    y_test = patch_labels[test_mask]

    # ----------------------------------------------
    # Normalize using training fold only
    # ----------------------------------------------
    (
        train_t1,
        train_t2,
        test_t1,
        test_t2
    ) = normalize_from_training(
        train_t1,
        train_t2,
        test_t1,
        test_t2
    )

    # ----------------------------------------------
    # NumPy -> PyTorch
    # ----------------------------------------------
    X1_train = torch.tensor(train_t1, dtype=torch.float32)
    X2_train = torch.tensor(train_t2, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)

    X1_test = torch.tensor(test_t1, dtype=torch.float32)
    X2_test = torch.tensor(test_t2, dtype=torch.float32)

    # ----------------------------------------------
    # Model
    # ----------------------------------------------
    net = SiameseChangeCNN()

    # compensate for class imbalance
    n_negative = np.sum(y_train == 0)
    n_positive = np.sum(y_train == 1)

    pos_weight = torch.tensor(
        [n_negative / n_positive],
        dtype=torch.float32
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        net.parameters(),
        lr=LEARNING_RATE
    )

    # ----------------------------------------------
    # Training
    # ----------------------------------------------
    net.train()

    for epoch in range(EPOCHS):

        optimizer.zero_grad()

        logits = net(
            X1_train,
            X2_train
        )

        loss = criterion(
            logits,
            y_train_t
        )

        loss.backward()
        optimizer.step()

    # ----------------------------------------------
    # Test inference
    # ----------------------------------------------
    net.eval()

    with torch.no_grad():

        test_logits = net(
            X1_test,
            X2_test
        )

        test_prob = torch.sigmoid(
            test_logits
        ).numpy()

    test_pred = (
        test_prob >= 0.5
    ).astype(int)

    return y_test, test_pred, test_prob


# ==================================================
# Fold 1
# Train upper -> test lower
# ==================================================

y_lower, pred_lower, prob_lower = train_and_test_fold(
    upper_fold,
    lower_fold,
    seed=42
)

# ==================================================
# Fold 2
# Train lower -> test upper
# ==================================================

y_upper, pred_upper, prob_upper = train_and_test_fold(
    lower_fold,
    upper_fold,
    seed=43
)

# ==================================================
# Combine spatial out-of-fold predictions
# ==================================================

y_all = np.concatenate([
    y_lower,
    y_upper
])

pred_all = np.concatenate([
    pred_lower,
    pred_upper
])

prob_all = np.concatenate([
    prob_lower,
    prob_upper
])

accuracy = accuracy_score(y_all, pred_all)
precision = precision_score(y_all, pred_all)
recall = recall_score(y_all, pred_all)
f1 = f1_score(y_all, pred_all)
auc = roc_auc_score(y_all, prob_all)

print("Spatial 2-fold Siamese CNN evaluation")
print("--------------------------------------")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")

print("\nFold-specific F1:")
print(
    "Upper → Lower:",
    round(f1_score(y_lower, pred_lower), 3)
)
print(
    "Lower → Upper:",
    round(f1_score(y_upper, pred_upper), 3)
)

In [ ]:
from sklearn.metrics import confusion_matrix, RocCurveDisplay
import matplotlib.pyplot as plt

# Confusion matrix
cm = confusion_matrix(y_all, pred_all)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(cm)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=14
        )

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels([
    "No urbanisation",
    "Urbanisation"
])

ax.set_yticklabels([
    "No urbanisation",
    "Urbanisation"
])

ax.set_xlabel("Predicted class")
ax.set_ylabel("Reference class")
ax.set_title(
    "Siamese CNN – Spatial 2-Fold Cross-Validation"
)

plt.tight_layout()
plt.show()


# ROC curve
RocCurveDisplay.from_predictions(
    y_all,
    prob_all
)

plt.title(
    f"Siamese CNN ROC Curve\nAUC = {auc:.3f}"
)

plt.show()

The fixed 0.50 threshold is used for the primary spatial 2-fold evaluation.  
A separate threshold of **0.58** is subsequently selected from the combined out-of-fold probabilities for mapping; its F1 is therefore an **optimised cross-validated value**, not an independent test estimate.

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.91, 0.01)

f1_values = []

for threshold in thresholds:
    pred = (prob_all >= threshold).astype(int)
    f1_values.append(
        f1_score(y_all, pred)
    )

best_index = np.argmax(f1_values)

best_threshold = thresholds[best_index]
best_f1 = f1_values[best_index]

print(
    "Best cross-validated threshold:",
    round(best_threshold, 2)
)

print(
    "F1 at this threshold:",
    round(best_f1, 3)
)

print(
    "F1 at threshold 0.50:",
    round(
        f1_score(
            y_all,
            (prob_all >= 0.5).astype(int)
        ),
        3
    )
)

In [ ]:
# ==================================================
# Final Siamese CNN trained on all reference patches
# ==================================================

torch.manual_seed(42)
np.random.seed(42)

# --------------------------------------------------
# Normalisation statistics from all labelled
# training patches
# --------------------------------------------------

combined_all = np.concatenate(
    [patches_t1, patches_t2],
    axis=0
)

final_band_mean = combined_all.mean(
    axis=(0, 2, 3),
    keepdims=True
)

final_band_std = combined_all.std(
    axis=(0, 2, 3),
    keepdims=True
)

final_band_std[
    final_band_std < 1e-6
] = 1.0

X1_all = (
    patches_t1 - final_band_mean
) / final_band_std

X2_all = (
    patches_t2 - final_band_mean
) / final_band_std

X1_all = torch.tensor(
    X1_all,
    dtype=torch.float32
)

X2_all = torch.tensor(
    X2_all,
    dtype=torch.float32
)

y_all_train = torch.tensor(
    patch_labels,
    dtype=torch.float32
)

# --------------------------------------------------
# Final model
# --------------------------------------------------

final_cnn = SiameseChangeCNN()

n_negative = np.sum(
    patch_labels == 0
)

n_positive = np.sum(
    patch_labels == 1
)

pos_weight = torch.tensor(
    [n_negative / n_positive],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    final_cnn.parameters(),
    lr=LEARNING_RATE
)

# --------------------------------------------------
# Train using the already fixed 60 epochs
# --------------------------------------------------

final_cnn.train()

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    logits = final_cnn(
        X1_all,
        X2_all
    )

    loss = criterion(
        logits,
        y_all_train
    )

    loss.backward()
    optimizer.step()

final_cnn.eval()

print("Final Siamese CNN trained.")
print(f"Final training loss: {loss.item():.4f}")

In [ ]:
# ==================================================
# Apply final Siamese CNN to the full Elmina area
# ==================================================

height, width = stack_t1.shape[1], stack_t1.shape[2]

cnn_probability_map = np.full(
    (height, width),
    np.nan,
    dtype=np.float32
)

final_cnn.eval()

with torch.no_grad():

    for row in range(HALF, height - HALF):
        for col in range(HALF, width - HALF):

            patch_t1 = stack_t1[
                :,
                row-HALF:row+HALF+1,
                col-HALF:col+HALF+1
            ]

            patch_t2 = stack_t2[
                :,
                row-HALF:row+HALF+1,
                col-HALF:col+HALF+1
            ]

            if (
                np.all(np.isfinite(patch_t1))
                and np.all(np.isfinite(patch_t2))
            ):

                # same normalization used for final training
                patch_t1_norm = (
                    patch_t1[None, ...] - final_band_mean
                ) / final_band_std

                patch_t2_norm = (
                    patch_t2[None, ...] - final_band_mean
                ) / final_band_std

                x1 = torch.tensor(
                    patch_t1_norm,
                    dtype=torch.float32
                )

                x2 = torch.tensor(
                    patch_t2_norm,
                    dtype=torch.float32
                )

                logit = final_cnn(
                    x1,
                    x2
                )

                probability = torch.sigmoid(
                    logit
                ).item()

                cnn_probability_map[
                    row,
                    col
                ] = probability


print(
    "Valid mapped pixels:",
    np.sum(np.isfinite(cnn_probability_map))
)

print(
    "Mean predicted urbanisation probability:",
    np.nanmean(cnn_probability_map)
)

In [ ]:
plt.figure(figsize=(8, 8))

im = plt.imshow(
    cnn_probability_map,
    cmap="viridis",
    vmin=0,
    vmax=1
)

plt.title(
    "Siamese CNN Urbanisation Probability\n"
    "Elmina, 2004–2025"
)

plt.colorbar(
    im,
    label="Predicted urbanisation probability"
)

plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
CNN_THRESHOLD = 0.58

cnn_final_mask = np.full(
    cnn_probability_map.shape,
    np.nan,
    dtype=np.float32
)

valid_cnn = np.isfinite(cnn_probability_map)

cnn_final_mask[valid_cnn] = (
    cnn_probability_map[valid_cnn] >= CNN_THRESHOLD
).astype(float)

cnn_percentage = (
    np.sum(cnn_final_mask[valid_cnn] == 1)
    / np.sum(valid_cnn)
    * 100
)

print(
    f"CNN detected urbanisation: "
    f"{cnn_percentage:.2f}% of mapped pixels"
)

cnn_display = np.ma.masked_invalid(cnn_final_mask)

cnn_cmap = ListedColormap([
    "lightgray",
    "firebrick"
])

fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(
    cnn_display,
    cmap=cnn_cmap,
    vmin=0,
    vmax=1
)

ax.set_title(
    "Siamese CNN Urbanisation Detection\n"
    "Elmina, 2004–2025 (threshold = 0.58)"
)

ax.axis("off")

ax.legend(
    handles=[
        Patch(facecolor="lightgray", label="No detected urbanisation change"),
        Patch(facecolor="firebrick", label="Detected urbanisation change")
    ],
    loc="lower left"
)

plt.tight_layout()
plt.show()


In [ ]:
# Common valid area for both methods
common_method_valid = (
    valid_cnn &
    valid_pixels
)

cnn_binary = (
    cnn_final_mask[common_method_valid] == 1
)

index_binary = (
    final_urbanisation_mask[common_method_valid]
)

# Overall agreement
agreement = np.mean(
    cnn_binary == index_binary
)

# Urbanisation percentages
cnn_share = (
    cnn_binary.mean() * 100
)

index_share = (
    index_binary.mean() * 100
)

# Jaccard / Intersection over Union for detected urbanisation
intersection = np.sum(
    cnn_binary & index_binary
)

union = np.sum(
    cnn_binary | index_binary
)

jaccard = intersection / union

print(
    f"Overall pixel agreement: {agreement:.3f}"
)

print(
    f"CNN urbanisation share: {cnn_share:.2f}%"
)

print(
    f"NDVI/NDBI urbanisation share: {index_share:.2f}%"
)

print(
    f"Urbanisation Jaccard index: {jaccard:.3f}"
)

### Siamese CNN result

With a fixed decision threshold of 0.50, the spatial 2-fold cross-validation achieved:

- Accuracy = **0.912**
- Precision = **0.875**
- Recall = **1.000**
- F1 = **0.933**
- ROC-AUC = **0.994**

Fold-specific F1 scores were 1.000 (upper → lower) and 0.894 (lower → upper).  
The CNN therefore provides a functioning locally supervised GeoAI approach, while also showing some spatial heterogeneity and a tendency toward false positives.

The final mapped CNN mask is more liberal than the NDVI/NDBI mask. Across their common valid area, the two methods show 75.0% pixel agreement and an urbanisation Jaccard index of 0.651.

## 7. Land Surface Temperature and robustness analysis

Absolute LST values from two individual acquisition dates are affected by the atmospheric and meteorological conditions of each date.  
Therefore, the analysis uses **scene-relative LST**: each pixel's LST is centred by the mean LST of its acquisition date before calculating change.

The resulting relative LST change is compared across:

1. the NDVI/NDBI baseline mask,
2. the Siamese CNN mask,
3. the manually labelled QGIS reference areas.

In [ ]:
def calculate_lst(item, bbox):
    """
    Calculate Landsat Collection 2 Level 2
    Land Surface Temperature in degrees Celsius.
    """

    signed_item = planetary_computer.sign(item)

    platform = item.properties.get("platform", "")

    if platform in ["landsat-4", "landsat-5", "landsat-7"]:
        thermal_asset = "lwir"
    elif platform in ["landsat-8", "landsat-9"]:
        thermal_asset = "lwir11"
    else:
        raise ValueError(
            f"Unsupported Landsat platform: {platform}"
        )

    url = signed_item.assets[thermal_asset].href

    with rasterio.open(url) as src:

        left, bottom, right, top = transform_bounds(
            "EPSG:4326",
            src.crs,
            bbox[0],
            bbox[1],
            bbox[2],
            bbox[3]
        )

        window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform
        )

        thermal_dn = src.read(
            1,
            window=window
        ).astype("float32")

    # Landsat fill values
    thermal_dn[thermal_dn == 0] = np.nan

    # Collection 2 Level 2 surface-temperature scaling
    temperature_kelvin = (
        thermal_dn * 0.00341802 + 149.0
    )

    temperature_celsius = (
        temperature_kelvin - 273.15
    )

    return temperature_celsius

In [ ]:
lst_t1 = calculate_lst(
    scene_t1,
    BBOX
)

lst_t2 = calculate_lst(
    scene_t2,
    BBOX
)

print("LST T1 shape:", lst_t1.shape)
print("LST T2 shape:", lst_t2.shape)

print(
    "Mean LST T1:",
    np.nanmean(lst_t1)
)

print(
    "Mean LST T2:",
    np.nanmean(lst_t2)
)

In [ ]:
common_valid = (
    np.isfinite(lst_t1) &
    np.isfinite(lst_t2)
)

mean_t1 = np.nanmean(lst_t1[common_valid])
mean_t2 = np.nanmean(lst_t2[common_valid])

relative_lst_t1 = np.where(
    common_valid,
    lst_t1 - mean_t1,
    np.nan
)

relative_lst_t2 = np.where(
    common_valid,
    lst_t2 - mean_t2,
    np.nan
)

relative_lst_change = (
    relative_lst_t2 -
    relative_lst_t1
)

vmin = np.nanpercentile(relative_lst_change, 2)
vmax = np.nanpercentile(relative_lst_change, 98)

norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0,
    vmax=vmax
)

plt.figure(figsize=(8, 8))

plt.imshow(
    relative_lst_change,
    norm=norm,
    cmap="coolwarm"
)

plt.colorbar(
    label="Relative LST change (°C)"
)

plt.title(
    "Relative LST Change\n"
    "Elmina, 2004–2025"
)

plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# NDVI/NDBI baseline
urban_lst_mask = (
    final_urbanisation_mask &
    common_valid
)

other_lst_mask = (
    (~final_urbanisation_mask) &
    common_valid
)

urban_relative_change = np.nanmean(
    relative_lst_change[urban_lst_mask]
)

other_relative_change = np.nanmean(
    relative_lst_change[other_lst_mask]
)

difference_relative = (
    urban_relative_change -
    other_relative_change
)

# Manual QGIS reference
reference_lst_valid = (
    labelled &
    common_valid
)

reference_urban = (
    reference_lst_valid &
    (reference == 1)
)

reference_nochange = (
    reference_lst_valid &
    (reference == 0)
)

reference_urban_change = np.nanmean(
    relative_lst_change[reference_urban]
)

reference_nochange_change = np.nanmean(
    relative_lst_change[reference_nochange]
)

reference_difference = (
    reference_urban_change -
    reference_nochange_change
)

print(f"NDVI/NDBI relative LST difference: {difference_relative:.3f} °C")
print(f"QGIS reference relative LST difference: {reference_difference:.3f} °C")

In [ ]:
# ==================================================
# Relative LST comparison using the Siamese CNN mask
# ==================================================

cnn_lst_valid = (
    valid_cnn &
    common_valid
)

cnn_urban_lst = (
    cnn_lst_valid &
    (cnn_final_mask == 1)
)

cnn_other_lst = (
    cnn_lst_valid &
    (cnn_final_mask == 0)
)

cnn_urban_relative_change = np.nanmean(
    relative_lst_change[cnn_urban_lst]
)

cnn_other_relative_change = np.nanmean(
    relative_lst_change[cnn_other_lst]
)

cnn_lst_difference = (
    cnn_urban_relative_change
    - cnn_other_relative_change
)

print(
    "CNN urbanisation relative LST change:",
    round(cnn_urban_relative_change, 3),
    "°C"
)

print(
    "CNN other areas relative LST change:",
    round(cnn_other_relative_change, 3),
    "°C"
)

print(
    "CNN relative LST difference:",
    round(cnn_lst_difference, 3),
    "°C"
)

print("\nPixel counts:")
print(
    "Urbanisation:",
    cnn_urban_lst.sum()
)

print(
    "Other:",
    cnn_other_lst.sum()
)

In [ ]:
methods = [
    "NDVI/NDBI",
    "Siamese CNN",
    "Manual QGIS reference"
]

lst_differences = [
    difference_relative,
    cnn_lst_difference,
    reference_difference
]

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    methods,
    lst_differences
)

ax.axhline(0, linewidth=1)

ax.set_ylabel(
    "Difference in relative LST change (°C)"
)

ax.set_title(
    "Relative LST Change Associated with Urbanisation\n"
    "Across Three Urbanisation Definitions"
)

ax.bar_label(
    bars,
    fmt="%.2f °C",
    padding=3
)

plt.tight_layout()
plt.show()


## 8. Final results summary

In [ ]:
summary_results = pd.DataFrame({
    "Result": [
        "NDVI/NDBI test-subset F1",
        "NDVI/NDBI test-subset accuracy",
        "ChangeStar ROC-AUC",
        "Siamese CNN spatial-CV F1 (threshold 0.50)",
        "Siamese CNN spatial-CV ROC-AUC",
        "Siamese CNN mapping threshold",
        "CNN–index pixel agreement",
        "CNN–index urbanisation Jaccard",
        "NDVI/NDBI relative LST difference",
        "Siamese CNN relative LST difference",
        "Manual QGIS relative LST difference"
    ],
    "Value": [
        f"{f1_test:.3f}",
        f"{accuracy_test:.3f}",
        f"{changestar_auc:.3f}",
        f"{f1:.3f}",
        f"{auc:.3f}",
        f"{CNN_THRESHOLD:.2f}",
        f"{agreement:.3f}",
        f"{jaccard:.3f}",
        f"{difference_relative:.2f} °C",
        f"{cnn_lst_difference:.2f} °C",
        f"{reference_difference:.2f} °C"
    ]
})

summary_results

## 9. Interpretation and limitations

The project finds a consistent association between urbanisation-related change and stronger scene-relative LST change in Elmina between the two Landsat observations.

Three urbanisation definitions produce the same direction of LST contrast:

- NDVI/NDBI baseline: approximately **+4.79 °C**
- Siamese CNN: approximately **+4.04 °C**
- manual QGIS reference: approximately **+6.91 °C**

The locally supervised Siamese CNN performs strongly under spatial 2-fold cross-validation (F1 = 0.933 at the fixed 0.50 threshold; ROC-AUC = 0.994). In contrast, the pretrained zero-shot ChangeStar transfer does not discriminate the urbanisation-specific reference target (ROC-AUC = 0.497), illustrating the importance of domain and target alignment in GeoAI.

Key limitations:

- only two Landsat acquisition dates are analysed;
- Landsat 5 and Landsat 9 are different sensors;
- the manually labelled reference sample is small and concentrated in four spatial blocks;
- 9×9 CNN patches overlap within each spatial block, so the 294 patches are not fully independent observations;
- the 0.58 CNN mapping threshold is optimised from out-of-fold predictions and should not be interpreted as an independent test result;
- scene-relative LST controls for the overall temperature level of each image but does not establish a causal urbanisation-induced temperature effect.

The results therefore support a robust **association**, not a causal estimate of urbanisation-induced warming.